# Step 5 — Continual Learning Strategies: EWC, Replay, PackNet

**Project:** Continual Self-Supervised Pretraining for Industrial Anomaly Localization

Implements and compares three CL mitigation strategies on the same
Bottle -> Cable -> Leather task stream from Notebook 4.
Each strategy is compared against the naive baseline BWT of -0.0633.

## 5.1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
assert torch.cuda.is_available()
print(f'GPU: {torch.cuda.get_device_name(0)}')
device = torch.device('cuda')

In [ ]:
import os, json, time, random
import numpy as np

PROJECT_ROOT = '/content/drive/MyDrive/anomaly_project'
DATA_ROOT    = f'{PROJECT_ROOT}/data/mvtec'
CKPT_ROOT    = f'{PROJECT_ROOT}/checkpoints'
RES_ROOT     = f'{PROJECT_ROOT}/results'
for d in [DATA_ROOT, CKPT_ROOT, RES_ROOT]: os.makedirs(d, exist_ok=True)

!pip install -q faiss-gpu-cu12 2>/dev/null || pip install -q faiss-cpu
print('Setup complete.')

## 5.2 — Shared infrastructure

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
import faiss
from sklearn.metrics import roc_auc_score
from scipy import ndimage
import pandas as pd
import matplotlib.pyplot as plt

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
TEXTURE_CATS  = {'leather', 'carpet', 'tile', 'wood', 'grid'}


class MVTecDataset(Dataset):
    def __init__(self, root, split='train', transform=None, mask_transform=None, two_crop=False):
        self.root = Path(root)
        self.split = split
        self.transform = transform
        self.mask_transform = mask_transform
        self.two_crop = two_crop
        self.samples = self._build_index()

    def _build_index(self):
        s = []
        if self.split == 'train':
            for p in sorted((self.root/'train'/'good').glob('*.png')):
                s.append({'image_path': p, 'label': 0, 'mask_path': None})
        elif self.split == 'test':
            for dd in sorted((self.root/'test').iterdir()):
                if not dd.is_dir(): continue
                is_anom = dd.name != 'good'
                for p in sorted(dd.glob('*.png')):
                    mp = (self.root/'ground_truth'/dd.name/f'{p.stem}_mask.png') if is_anom else None
                    s.append({'image_path': p, 'label': int(is_anom),
                               'mask_path': mp, 'defect_type': dd.name})
        return s

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s['image_path']).convert('RGB')
        if self.split == 'train':
            if self.two_crop: return self.transform(img), self.transform(img)
            return self.transform(img)
        img_t = self.transform(img)
        if s['mask_path'] is not None:
            mask = Image.open(s['mask_path']).convert('L')
            mt = self.mask_transform(mask) if self.mask_transform else mask
            mt = torch.from_numpy((np.array(mt) > 0).astype(np.float32))
        else:
            h, w = img_t.shape[-2:]
            mt = torch.zeros((h, w), dtype=torch.float32)
        return {'image': img_t, 'label': s['label'], 'mask': mt,
                'defect_type': s['defect_type'], 'image_path': str(s['image_path'])}


def get_simclr_transform(cat):
    s = 0.2
    hue = 0.0 if cat in TEXTURE_CATS else s * 0.5
    scale_lo = 0.2 if cat in TEXTURE_CATS else 0.5
    return T.Compose([
        T.RandomResizedCrop(IMG_SIZE, scale=(scale_lo, 1.0)),
        T.RandomHorizontalFlip(0.5),
        T.ColorJitter(s, s, s, hue),
        T.GaussianBlur(3, (0.1, 1.0)),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
mask_tf = T.Resize((IMG_SIZE, IMG_SIZE))


class SimCLRHead(nn.Module):
    def __init__(self, in_dim=512, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, in_dim), nn.ReLU(True), nn.Linear(in_dim, out_dim))
    def forward(self, x): return self.net(x)


def nt_xent(z_a, z_b, temp=0.5):
    N = z_a.shape[0]
    z = F.normalize(torch.cat([z_a, z_b]), dim=1)
    sim = torch.matmul(z, z.T) / temp
    sim.masked_fill_(torch.eye(2*N, dtype=torch.bool, device=z.device), float('-inf'))
    pos = torch.cat([torch.arange(N, 2*N, device=z.device), torch.arange(N, device=z.device)])
    return F.cross_entropy(sim, pos)


print('Shared dataset, transforms, NT-Xent ready.')

In [ ]:
class PatchExtractor(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.bb = backbone
        self.bb.eval()
        for p in self.bb.parameters(): p.requires_grad = False
        self._f = {}
        for name in ['layer1', 'layer2', 'layer3']:
            getattr(backbone, name).register_forward_hook(
                lambda m, i, o, n=name: self._f.update({n: o})
            )

    @torch.no_grad()
    def forward(self, x):
        self._f = {}
        self.bb(x)
        f1, f2, f3 = self._f['layer1'], self._f['layer2'], self._f['layer3']
        kw = dict(mode='bilinear', align_corners=False)
        return torch.cat([f1, F.interpolate(f2, f1.shape[-2:], **kw),
                           F.interpolate(f3, f1.shape[-2:], **kw)], dim=1)


def score_and_evaluate(backbone, category):
    ext = PatchExtractor(backbone).to(device)
    # Build memory bank
    ds = MVTecDataset(f'{DATA_ROOT}/{category}', split='train', transform=eval_tf)
    ldr = DataLoader(ds, batch_size=16, shuffle=False, num_workers=2)
    patches, gsize = [], None
    with torch.no_grad():
        for b in ldr:
            fm = ext(b.to(device)); B, C, H, W = fm.shape; gsize = (H, W)
            patches.append(fm.permute(0,2,3,1).reshape(-1,C).cpu())
    patches = torch.cat(patches)
    tgt = int(len(patches) * 0.10)
    feats = patches.to(device)
    sel = [torch.randint(0, len(feats), (1,)).item()]
    md = torch.cdist(feats, feats[sel]).squeeze(1)
    for _ in range(tgt - 1):
        nx = torch.argmax(md).item(); sel.append(nx)
        md = torch.minimum(md, torch.cdist(feats, feats[nx:nx+1]).squeeze(1))
    bank = feats[sel].cpu()
    # Score test set
    idx = faiss.IndexFlatL2(bank.shape[1]); idx.add(bank.numpy().astype('float32'))
    H, W = gsize
    tds = MVTecDataset(f'{DATA_ROOT}/{category}', split='test', transform=eval_tf, mask_transform=mask_tf)
    tldr = DataLoader(tds, batch_size=8, shuffle=False, num_workers=2)
    sc, lab, am, gt = [], [], [], []
    with torch.no_grad():
        for batch in tldr:
            fm = ext(batch['image'].to(device)); B, C, _, _ = fm.shape
            ps = fm.permute(0,2,3,1).reshape(B, H*W, C).cpu().numpy().astype('float32')
            for b in range(B):
                d, _ = idx.search(ps[b], k=1); pd = d[:,0].reshape(H,W)
                sc.append(pd.max())
                up = F.interpolate(torch.from_numpy(pd).unsqueeze(0).unsqueeze(0).float(),
                                   (IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False).squeeze().numpy()
                am.append(up)
            lab.extend(batch['label'].tolist()); gt.extend([m.numpy() for m in batch['mask']])
    auroc = roc_auc_score(np.array(lab), np.array(sc))
    # PRO
    all_s = np.concatenate([m.flatten() for m in am])
    ths = np.unique(np.percentile(all_s, np.linspace(0,100,50)))
    lms = []
    for m in gt:
        if m.sum() > 0: l, n = ndimage.label(m); lms.append((l, n))
        else: lms.append((None, 0))
    pros, fprs = [], []
    tnp = sum((m==0).sum() for m in gt)
    for th in ths:
        rtprs, fp = [], 0
        for a, m, (l, n) in zip(am, gt, lms):
            p = (a >= th); fp += np.logical_and(p, m==0).sum()
            if n > 0:
                for r in range(1, n+1):
                    rm = (l == r)
                    if rm.sum() == 0: continue
                    rtprs.append(np.logical_and(p, rm).sum() / rm.sum())
        pros.append(np.mean(rtprs) if rtprs else 0.0); fprs.append(fp / max(tnp, 1))
    order = np.argsort(fprs); fs = np.array(fprs)[order]; ps = np.array(pros)[order]
    mr = fs <= 0.3
    if mr.sum() < 2: return auroc, 0.0
    trapz = getattr(np, 'trapezoid', None) or np.trapz
    return auroc, trapz(ps[mr], fs[mr]) / 0.3


def save_ckpt(state, cat, strat, tag='stream'):
    d = f'{CKPT_ROOT}/{cat}'; os.makedirs(d, exist_ok=True)
    path = f'{d}/{strat}_{tag}.pt'; state['_t'] = time.time()
    torch.save(state, path); return path


def load_ckpt(cat, strat, tag='stream'):
    path = f'{CKPT_ROOT}/{cat}/{strat}_{tag}.pt'
    return torch.load(path, map_location=device) if os.path.exists(path) else None


print('PatchExtractor, score_and_evaluate, checkpointing ready.')

## 5.3 — EWC implementation

In [ ]:
def compute_fisher(backbone, proj_head, loader, n_batches=20):
    backbone.train(); proj_head.train()
    fisher = {n: torch.zeros_like(p, device=device) for n, p in backbone.named_parameters()}
    seen = 0
    for i, (va, vb) in enumerate(loader):
        if i >= n_batches: break
        va, vb = va.to(device), vb.to(device)
        backbone.zero_grad(); proj_head.zero_grad()
        nt_xent(proj_head(backbone(va)), proj_head(backbone(vb))).backward()
        for n, p in backbone.named_parameters():
            if p.grad is not None: fisher[n] += p.grad.detach() ** 2
        seen += 1
    for n in fisher: fisher[n] /= max(seen, 1)
    return fisher


def ewc_loss(backbone, fishers, theta_stars, lam=100.0):
    loss = torch.tensor(0.0, device=device)
    for f, ts in zip(fishers, theta_stars):
        for n, p in backbone.named_parameters():
            if n in f: loss = loss + (f[n] * (p - ts[n].to(device)) ** 2).sum()
    return lam / 2.0 * loss


def finetune_ewc(backbone, cat, epochs, fishers, theta_stars, lam=100.0, bs=64, lr=3e-4):
    ds = MVTecDataset(f'{DATA_ROOT}/{cat}', split='train',
                       transform=get_simclr_transform(cat), two_crop=True)
    ldr = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    proj = SimCLRHead().to(device); backbone.train()
    opt = torch.optim.Adam(list(backbone.parameters()) + list(proj.parameters()), lr=lr)
    hist = []
    for ep in range(epochs):
        losses = []
        for va, vb in ldr:
            va, vb = va.to(device), vb.to(device)
            loss = nt_xent(proj(backbone(va)), proj(backbone(vb))) + ewc_loss(backbone, fishers, theta_stars, lam)
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        hist.append(sum(losses)/len(losses))
        if ep % 50 == 0 or ep == epochs-1:
            print(f'    epoch {ep:3d}/{epochs} | loss = {hist[-1]:.4f}')
    backbone.eval()
    f_new = compute_fisher(backbone, proj, ldr)
    ts_new = {n: p.detach().clone() for n, p in backbone.named_parameters()}
    return backbone, hist, f_new, ts_new


print('EWC functions defined.')

## 5.4 — Coreset Replay implementation

In [ ]:
def build_replay_buffer(cat, n=100, seed=42):
    ds = MVTecDataset(f'{DATA_ROOT}/{cat}', split='train',
                       transform=get_simclr_transform(cat), two_crop=True)
    rng = random.Random(seed)
    idx = rng.sample(range(len(ds)), min(n, len(ds)))
    buf = [ds[i] for i in idx]
    print(f'  Replay buffer for {cat!r}: {len(buf)} pairs.')
    return buf


def finetune_replay(backbone, cat, epochs, replay_buffers, n_replay=32, bs=64, lr=3e-4):
    ds = MVTecDataset(f'{DATA_ROOT}/{cat}', split='train',
                       transform=get_simclr_transform(cat), two_crop=True)
    ldr = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    combined = [item for buf in replay_buffers.values() for item in buf]
    proj = SimCLRHead().to(device); backbone.train()
    opt = torch.optim.Adam(list(backbone.parameters()) + list(proj.parameters()), lr=lr)
    hist = []
    for ep in range(epochs):
        losses = []
        for ca, cb in ldr:
            ca, cb = ca.to(device), cb.to(device)
            if combined:
                ri = random.sample(range(len(combined)), min(n_replay, len(combined)))
                ra = torch.stack([combined[i][0] for i in ri]).to(device)
                rb = torch.stack([combined[i][1] for i in ri]).to(device)
                ca, cb = torch.cat([ca, ra]), torch.cat([cb, rb])
            loss = nt_xent(proj(backbone(ca)), proj(backbone(cb)))
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        hist.append(sum(losses)/len(losses))
        if ep % 50 == 0 or ep == epochs-1:
            print(f'    epoch {ep:3d}/{epochs} | loss = {hist[-1]:.4f}')
    backbone.eval()
    return backbone, hist


print('Replay functions defined.')

## 5.5 — PackNet implementation

In [ ]:
def update_masks(backbone, existing_masks, prune_ratio=0.5):
    new_masks = {}
    for n, p in backbone.named_parameters():
        free = ~existing_masks[n].bool() if n in existing_masks else torch.ones_like(p, dtype=torch.bool)
        fw = p.detach().abs()[free]
        if fw.numel() == 0:
            new_masks[n] = existing_masks.get(n, torch.zeros_like(p)); continue
        th = torch.quantile(fw, 1.0 - prune_ratio)
        freeze = free & (p.detach().abs() >= th)
        new_masks[n] = (existing_masks[n].bool() | freeze).float() if n in existing_masks else freeze.float()
    return new_masks


def mask_grads(backbone, masks):
    for n, p in backbone.named_parameters():
        if n in masks and p.grad is not None:
            p.grad *= (1.0 - masks[n].to(p.device))


def finetune_packnet(backbone, cat, epochs, masks, prune_ratio=0.5, bs=64, lr=3e-4):
    ds = MVTecDataset(f'{DATA_ROOT}/{cat}', split='train',
                       transform=get_simclr_transform(cat), two_crop=True)
    ldr = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    proj = SimCLRHead().to(device); backbone.train()
    opt = torch.optim.Adam(list(backbone.parameters()) + list(proj.parameters()), lr=lr)
    hist = []
    for ep in range(epochs):
        losses = []
        for va, vb in ldr:
            va, vb = va.to(device), vb.to(device)
            loss = nt_xent(proj(backbone(va)), proj(backbone(vb)))
            opt.zero_grad(); loss.backward()
            mask_grads(backbone, masks)
            opt.step()
            losses.append(loss.item())
        hist.append(sum(losses)/len(losses))
        if ep % 50 == 0 or ep == epochs-1:
            print(f'    epoch {ep:3d}/{epochs} | loss = {hist[-1]:.4f}')
    backbone.eval()
    new_masks = update_masks(backbone, masks, prune_ratio)
    frozen = sum((m==1).float().mean().item() for m in new_masks.values()) / len(new_masks)
    print(f'    PackNet: {frozen:.1%} of parameters frozen after this task.')
    return backbone, hist, new_masks


print('PackNet functions defined.')

## 5.6 — Shared stream runner and BWT

In [ ]:
TASK_STREAM    = ['bottle', 'cable', 'leather']
EPOCHS_PER_TASK = 300
NAIVE_BWT      = -0.0633  # from Notebook 4


def fresh_imagenet_backbone():
    bb = models.resnet18(weights=None); bb.fc = nn.Identity()
    imgnet = models.resnet18(weights='IMAGENET1K_V1')
    bb.load_state_dict({k: v for k, v in imgnet.state_dict().items()
                        if k not in ('fc.weight', 'fc.bias')}, strict=False)
    return bb.to(device)


def run_stream(strategy_name, finetune_fn):
    state_path = f'{RES_ROOT}/phase2_{strategy_name}_state.json'
    if os.path.exists(state_path):
        with open(state_path) as f: ss = json.load(f)
        print(f'Resuming {strategy_name!r}: done={ss["completed_tasks"]}')
    else:
        ss = {'completed_tasks': [], 'records': []}
        print(f'Starting {strategy_name!r} from scratch.')

    bb = models.resnet18(weights=None); bb.fc = nn.Identity()
    if ss['completed_tasks']:
        last = ss['completed_tasks'][-1]
        ckpt = load_ckpt(last, strategy_name)
        bb.load_state_dict(ckpt['backbone_state'])
        print(f'Loaded backbone from after task {last!r}.')
    else:
        imgnet = models.resnet18(weights='IMAGENET1K_V1')
        bb.load_state_dict({k: v for k, v in imgnet.state_dict().items()
                            if k not in ('fc.weight', 'fc.bias')}, strict=False)
        print('Starting from ImageNet weights.')
    bb = bb.to(device)
    cl_state = {}

    for ti, cat in enumerate(TASK_STREAM):
        if cat in ss['completed_tasks']:
            print(f'Task {cat!r} already done -- skipping.'); continue
        print(f"\n{'='*56}\nTask {ti+1}/{len(TASK_STREAM)}: {cat!r} [{strategy_name}]\n{'='*56}")
        bb, cl_state = finetune_fn(bb, cat, EPOCHS_PER_TASK, cl_state)
        save_ckpt({'backbone_state': bb.state_dict(), 'task': cat}, cat, strategy_name)
        seen = TASK_STREAM[:ti+1]
        print(f'Evaluating on: {seen}')
        for ec in seen:
            auroc, pro = score_and_evaluate(bb, ec)
            print(f'  eval {ec!r}: AUROC={auroc:.4f} | PRO={pro:.4f}')
            ss['records'].append({'strategy': strategy_name, 'trained_through': cat,
                                   'trained_through_idx': ti, 'eval_category': ec,
                                   'auroc': float(auroc), 'pro': float(pro)})
        ss['completed_tasks'].append(cat)
        with open(state_path, 'w') as f: json.dump(ss, f, indent=2)
        print(f'State saved ({len(ss["completed_tasks"])}/{len(TASK_STREAM)} done).')

    return ss['records']


def compute_bwt(records, strategy):
    df = pd.DataFrame([r for r in records if r['strategy'] == strategy])
    fi = len(TASK_STREAM) - 1; terms = []
    for i, cat in enumerate(TASK_STREAM[:-1]):
        jl = df[(df['eval_category']==cat) & (df['trained_through_idx']==i)]['auroc'].iloc[0]
        ae = df[(df['eval_category']==cat) & (df['trained_through_idx']==fi)]['auroc'].iloc[0]
        terms.append(ae - jl)
    return sum(terms) / len(terms)


print('run_stream() and compute_bwt() defined.')

## 5.7 — Run EWC

In [ ]:
EWC_LAMBDA = 100.0


def ewc_fn(bb, cat, epochs, state):
    bb, hist, f, ts = finetune_ewc(bb, cat, epochs,
                                    state.get('fishers', []), state.get('thetas', []),
                                    lam=EWC_LAMBDA)
    state['fishers'] = state.get('fishers', []) + [f]
    state['thetas']  = state.get('thetas', [])  + [ts]
    return bb, state


ewc_records = run_stream('ewc', ewc_fn)
ewc_bwt = compute_bwt(ewc_records, 'ewc')
print(f'\nEWC BWT: {ewc_bwt:+.4f}  (naive: {NAIVE_BWT:+.4f})')
with open(f'{RES_ROOT}/phase2_ewc_summary.json', 'w') as f:
    json.dump({'strategy': 'ewc', 'lambda': EWC_LAMBDA, 'bwt': ewc_bwt, 'records': ewc_records}, f, indent=2)

## 5.8 — Run Coreset Replay

In [ ]:
REPLAY_N = 100


def replay_fn(bb, cat, epochs, state):
    bufs = state.get('buffers', {})
    bb, hist = finetune_replay(bb, cat, epochs, bufs, n_replay=32)
    bufs[cat] = build_replay_buffer(cat, n=REPLAY_N)
    state['buffers'] = bufs
    return bb, state


replay_records = run_stream('replay', replay_fn)
replay_bwt = compute_bwt(replay_records, 'replay')
print(f'\nReplay BWT: {replay_bwt:+.4f}  (naive: {NAIVE_BWT:+.4f})')
with open(f'{RES_ROOT}/phase2_replay_summary.json', 'w') as f:
    json.dump({'strategy': 'replay', 'buffer_size': REPLAY_N, 'bwt': replay_bwt,
               'records': replay_records}, f, indent=2)

## 5.9 — Run PackNet

In [ ]:
PRUNE_RATIO = 0.50


def packnet_fn(bb, cat, epochs, state):
    bb, hist, masks = finetune_packnet(bb, cat, epochs, state.get('masks', {}),
                                        prune_ratio=PRUNE_RATIO)
    state['masks'] = masks
    return bb, state


packnet_records = run_stream('packnet', packnet_fn)
packnet_bwt = compute_bwt(packnet_records, 'packnet')
print(f'\nPackNet BWT: {packnet_bwt:+.4f}  (naive: {NAIVE_BWT:+.4f})')
with open(f'{RES_ROOT}/phase2_packnet_summary.json', 'w') as f:
    json.dump({'strategy': 'packnet', 'prune_ratio': PRUNE_RATIO, 'bwt': packnet_bwt,
               'records': packnet_records}, f, indent=2)

## 5.10 — Phase 2 comparison: all strategies side by side

In [ ]:
with open(f'{RES_ROOT}/phase2_naive_continual_summary.json') as f:
    naive_data = json.load(f)
naive_records = [{**r, 'strategy': 'naive'} for r in naive_data['records']]

all_records = naive_records + ewc_records + replay_records + packnet_records
all_df = pd.DataFrame(all_records)

bwt_df = pd.DataFrame([
    {'strategy': 'naive',   'bwt': NAIVE_BWT},
    {'strategy': 'ewc',     'bwt': ewc_bwt},
    {'strategy': 'replay',  'bwt': replay_bwt},
    {'strategy': 'packnet', 'bwt': packnet_bwt},
])
print('=== Phase 2 BWT comparison ===')
print(bwt_df.to_string(index=False))
print('(less negative = less forgetting; naive is the baseline)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
strats  = ['naive', 'ewc', 'replay', 'packnet']
colors  = ['gray', 'steelblue', 'darkorange', 'forestgreen']

for eval_cat, ax in zip(['bottle', 'cable'], axes):
    for s, c in zip(strats, colors):
        sub = all_df[(all_df['strategy']==s) & (all_df['eval_category']==eval_cat)]
        if sub.empty: continue
        ax.plot(sub['trained_through_idx'], sub['auroc'], marker='o', label=s, color=c)
    ax.set_title(f'Forgetting curve -- {eval_cat}')
    ax.set_xlabel('Task index')
    ax.set_ylabel('AUROC')
    ax.set_xticks(range(len(TASK_STREAM)))
    ax.set_xticklabels(TASK_STREAM)
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Phase 2: CL strategy comparison', fontsize=13)
plt.tight_layout(); plt.show()

bwt_df.to_csv(f'{RES_ROOT}/phase2_bwt_comparison.csv', index=False)
print(f'Saved to {RES_ROOT}/phase2_bwt_comparison.csv')

## Step 5 complete

- [ ] EWC, Replay, PackNet each run on the full Bottle -> Cable -> Leather stream
- [ ] BWT recorded and compared against the naive baseline (-0.0633)
- [ ] Forgetting curves plotted for all four strategies
- [ ] Results saved to Drive under `results/phase2_*_summary.json`

**This completes Phase 2 (RQ2).**